In [14]:

pip install cloudscraper


   ---------------------------------------- 2/2 [cloudscraper]

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import io

url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_squads"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(url, headers=headers)
html = response.text
soup = BeautifulSoup(html, 'html.parser')

all_squads = []

for table in soup.find_all('table', class_='sortable'):
    header = table.find_previous(['h3', 'h2'])
    team_name = header.text.replace('[edit]', '').strip() if header else 'Unknown'
    
    try:
        df = pd.read_html(io.StringIO(str(table)))[0]
    except Exception:
        continue
        
    if 'Player' in df.columns and 'Club' in df.columns:
        df['Team'] = team_name
        all_squads.append(df)

if all_squads:
    full_squads_df = pd.concat(all_squads, ignore_index=True)
    
    # Drop rows that have missing values in more than 3 fields
    min_non_na = full_squads_df.shape[1] - 3
    full_squads_df.dropna(thresh=min_non_na, inplace=True)
    full_squads_df.reset_index(drop=True, inplace=True)
    
    print(f"Loaded {len(full_squads_df)} players across {len(all_squads)} teams.")
else:
    print("No squads found.")

    

Loaded 1240 players across 44 teams.


,No.,Pos.,Player,Date of birth (age),Caps,Goals,Club,Team
0,NaN,GK,Matěj Kovář,"May 17, 2000 (aged 26)",19.0,0.0,PSV Eindhoven,Czech Republic
1,NaN,GK,Jindřich Staněk,"April 27, 1996 (aged 30)",14.0,0.0,Slavia Prague,Czech Republic
2,NaN,GK,Lukáš Horníček,"July 13, 2002 (aged 23)",0.0,0.0,Braga,Czech Republic
3,NaN,DF,Vladimír Coufal,"August 22, 1992 (aged 33)",61.0,2.0,TSG Hoffenheim,Czech Republic
4,NaN,DF,Tomáš Holeš,"March 31, 1993 (aged 33)",39.0,2.0,Slavia Prague,Czech Republic
5,NaN,DF,Ladislav Krejčí (captain),"April 20, 1999 (aged 27)",25.0,5.0,Wolverhampton Wanderers,Czech Republic
6,NaN,DF,David Zima,"November 8, 2000 (aged 25)",24.0,1.0,Slavia Prague,Czech Republic
7,NaN,DF,Jaroslav Zelený,"August 20, 1992 (aged 33)",21.0,0.0,Sparta Prague,Czech Republic
8,NaN,DF,David Jurásek,"August 7, 2000 (aged 25)",16.0,1.0,Slavia Prague,Czech Republic
9,NaN,DF,David Douděra,"May 31, 1998 (aged 28)",15.0,2.0,Slavia Prague,Czech Republic


### Save DataFrame to CSV File

In [ ]:
# Process captain designation and clean player names
import re

# Create Is_Captain column (1 if captain, 0 otherwise)
full_squads_df['Is_Captain'] = full_squads_df['Player'].str.contains('(captain)', case=False, na=False).astype(int)

# Remove (captain) from player names
full_squads_df['Player'] = full_squads_df['Player'].str.replace(r'\s*\(captain\)\s*', '', flags=re.IGNORECASE, regex=True)

# Save updated CSV
storage_path = "world_cup_squadsNew.csv"
full_squads_df.to_csv(storage_path, index=False)

print(f"✓ Updated {full_squads_df['Is_Captain'].sum()} captains identified")
print(f"✓ Player names cleaned and Is_Captain column added")
print(f"✓ File saved to {storage_path}")
print(f"\nFirst few rows:")
print(full_squads_df[['Player', 'Club', 'Team', 'Is_Captain']].head(10))


✓ Updated 44 captains identified
✓ Player names cleaned and Is_Captain column added
✓ File saved to world_cup_squads.csv

First few rows:
            Player                     Club            Team  Is_Captain
0      Matěj Kovář            PSV Eindhoven  Czech Republic           0
1  Jindřich Staněk            Slavia Prague  Czech Republic           0
2   Lukáš Horníček                    Braga  Czech Republic           0
3  Vladimír Coufal           TSG Hoffenheim  Czech Republic           0
4      Tomáš Holeš            Slavia Prague  Czech Republic           0
5  Ladislav Krejčí  Wolverhampton Wanderers  Czech Republic           1
6       David Zima            Slavia Prague  Czech Republic           0
7  Jaroslav Zelený            Sparta Prague  Czech Republic           0
8    David Jurásek            Slavia Prague  Czech Republic           0
9    David Douděra            Slavia Prague  Czech Republic           0


C:\Users\sraul\AppData\Local\Temp\ipykernel_20004\786011682.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  full_squads_df['Is_Captain'] = full_squads_df['Player'].str.contains('(captain)', case=False, na=False).astype(int)


In [27]:


storage_path = "world_cup_squads.csv"
full_squads_df.to_csv(storage_path, index=False)